In [1]:
import pandas as pd    # to load dataset
import numpy as np     # for mathematic equation
from nltk.corpus import stopwords   # to get collection of stopwords
from sklearn.model_selection import train_test_split       # for splitting dataset
from tensorflow.keras.preprocessing.text import Tokenizer  # to encode text to int
from tensorflow.keras.preprocessing.sequence import pad_sequences   # to do padding or truncating
from tensorflow.keras.models import Sequential     # the model
from tensorflow.keras.layers import Embedding, LSTM, Dense # layers of the architecture
from tensorflow.keras.callbacks import ModelCheckpoint   # save model
from tensorflow.keras.models import load_model   # load saved model

In [2]:
data = pd.read_csv('IMDB Dataset.csv')

print(data)

                                                  review sentiment
0      One of the other reviewers has mentioned that ...  positive
1      A wonderful little production. <br /><br />The...  positive
2      I thought this was a wonderful way to spend ti...  positive
3      Basically there's a family where a little boy ...  negative
4      Petter Mattei's "Love in the Time of Money" is...  positive
...                                                  ...       ...
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

[50000 rows x 2 columns]


In [8]:
english_stops = set(stopwords.words('english'))

In [9]:
def load_dataset():
    df = pd.read_csv('IMDB Dataset.csv')
    x_data = df['review']       # Reviews/Input
    y_data = df['sentiment']    # Sentiment/Output

    # PRE-PROCESS REVIEW
    x_data = x_data.replace({'<.*?>': ''}, regex = True)          # remove html tag
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex = True)     # remove non alphabet
    x_data = x_data.apply(lambda review: [w for w in review.split() if w not in english_stops])  # remove stop words
    x_data = x_data.apply(lambda review: [w.lower() for w in review])   # lower case

    # ENCODE SENTIMENT -> 0 & 1
    y_data = y_data.replace('positive', 1)
    y_data = y_data.replace('negative', 0)

    return x_data, y_data

x_data, y_data = load_dataset()

print('Reviews')
print(x_data, '\n')
print('Sentiment')
print(y_data)

Reviews
0        [one, reviewers, mentioned, watching, oz, epis...
1        [a, wonderful, little, production, the, filmin...
2        [i, thought, wonderful, way, spend, time, hot,...
3        [basically, family, little, boy, jake, thinks,...
4        [petter, mattei, love, time, money, visually, ...
                               ...                        
49995    [i, thought, movie, right, good, job, it, crea...
49996    [bad, plot, bad, dialogue, bad, acting, idioti...
49997    [i, catholic, taught, parochial, elementary, s...
49998    [i, going, disagree, previous, comment, side, ...
49999    [no, one, expects, star, trek, movies, high, a...
Name: review, Length: 50000, dtype: object 

Sentiment
0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 50000, dtype: int64


C:\Users\Harsh\AppData\Local\Temp\ipykernel_42480\4067306172.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y_data = y_data.replace('negative', 0)


In [10]:
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size = 0.2)

print('Train Set')
print(x_train, '\n')
print(x_test, '\n')
print('Test Set')
print(y_train, '\n')
print(y_test)

Train Set
22810    [govind, nihalani, directorial, venture, vijay...
22561    [perhaps, i, really, good, mood, i, watched, f...
13099    [conrad, radzoff, ferdy, mayne, hammy, cult, i...
10684    [let, face, rented, stdvd, sequel, forgotten, ...
13260    [my, sisters, cousin, female, forced, see, chi...
                               ...                        
31772    [like, curse, of, the, komodo, creature, featu...
46846    [i, going, afi, list, top, comedies, i, must, ...
18510    [just, let, everyone, know, possibly, worst, m...
20081    [i, could, believe, awful, film, i, rarely, wa...
26041    [there, time, alien, series, success, even, th...
Name: review, Length: 40000, dtype: object 

30154    [we, congratulate, uwe, boll, he, done, unthin...
46245    [welcome, oakland, dead, come, play, even, boy...
41963    [believe, movie, terrible, waste, time, would,...
4670     [besides, planes, trains, automobiles, uncle, ...
17906    [i, caught, first, time, nights, ago, televisi...
 

In [11]:
def get_max_length():
    review_length = []
    for review in x_train:
        review_length.append(len(review))

    return int(np.ceil(np.mean(review_length)))

In [12]:
# ENCODE REVIEW
token = Tokenizer(lower=False)    # no need lower, because already lowered the data in load_data()
token.fit_on_texts(x_train)
x_train = token.texts_to_sequences(x_train)
x_test = token.texts_to_sequences(x_test)

max_length = get_max_length()

x_train = pad_sequences(x_train, maxlen=max_length, padding='post', truncating='post')
x_test = pad_sequences(x_test, maxlen=max_length, padding='post', truncating='post')

total_words = len(token.word_index) + 1   # add 1 because of 0 padding

print('Encoded X Train\n', x_train, '\n')
print('Encoded X Test\n', x_test, '\n')
print('Maximum review length: ', max_length)

Encoded X Train
 [[28154 30154  3590 ...     0     0     0]
 [  292     1    14 ...     0     0     0]
 [ 4348 18201 26445 ...     0     0     0]
 ...
 [  436   183   197 ...     0     0     0]
 [    1    27   167 ...     0     0     0]
 [   50    10  1072 ... 16449  3084   515]] 

Encoded X Test
 [[  223 15624  5446 ...    31   224   249]
 [ 2454 16196   249 ...  4644  2577 33621]
 [  167     3   285 ...     0     0     0]
 ...
 [    1  2995  1128 ...     0     0     0]
 [  291    55   302 ...     0     0     0]
 [  366    38    20 ...   388     1    31]] 

Maximum review length:  130


In [13]:
# ARCHITECTURE
EMBED_DIM = 32
LSTM_OUT = 64

model = Sequential()
model.add(Embedding(total_words, EMBED_DIM, input_length = max_length))
model.add(LSTM(LSTM_OUT))
model.add(Dense(1, activation='sigmoid'))
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

print(model.summary())


Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 130, 32)           2958944   
                                                                 
 lstm (LSTM)                 (None, 64)                24832     
                                                                 
 dense (Dense)               (None, 1)                 65        
                                                                 
Total params: 2983841 (11.38 MB)
Trainable params: 2983841 (11.38 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
None


In [14]:
checkpoint = ModelCheckpoint(
    'models/model.h5',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

In [15]:
model.fit(x_train, y_train, batch_size = 128, epochs = 5, callbacks=[checkpoint])

Epoch 1/5


313/313 [==============================] - ETA: 0s - loss: 0.4738 - accuracy: 0.7386
Epoch 1: accuracy improved from -inf to 0.73855, saving model to models\model.h5
313/313 [==============================] - 35s 106ms/step - loss: 0.4738 - accuracy: 0.7386
Epoch 2/5


c:\Users\Harsh\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


313/313 [==============================] - ETA: 0s - loss: 0.2099 - accuracy: 0.9241
Epoch 2: accuracy improved from 0.73855 to 0.92413, saving model to models\model.h5
313/313 [==============================] - 30s 97ms/step - loss: 0.2099 - accuracy: 0.9241
Epoch 3/5
313/313 [==============================] - ETA: 0s - loss: 0.1238 - accuracy: 0.9610
Epoch 3: accuracy improved from 0.92413 to 0.96105, saving model to models\model.h5
313/313 [==============================] - 30s 97ms/step - loss: 0.1238 - accuracy: 0.9610
Epoch 4/5
313/313 [==============================] - ETA: 0s - loss: 0.0772 - accuracy: 0.9778
Epoch 4: accuracy improved from 0.96105 to 0.97777, saving model to models\model.h5
313/313 [==============================] - 30s 97ms/step - loss: 0.0772 - accuracy: 0.9778
Epoch 5/5
313/313 [==============================] - ETA: 0s - loss: 0.0605 - accuracy: 0.9839
Epoch 5: accuracy improved from 0.97777 to 0.98392, saving model to models\model.h5
313/313 [============

In [18]:
y_pred = (model.predict(x_test, batch_size=128) > 0.5).astype("int32")

total = len(y_test)
true = 0
for i, y in enumerate(y_test):
    if y == y_pred[i][0]:
        true += 1

print('Correct Prediction: {}'.format(true))
print('Wrong Prediction: {}'.format(total - true))
print('Accuracy: {}'.format(true/total*100))

79/79 [==============================] - 3s 27ms/step
Correct Prediction: 8673
Wrong Prediction: 1327
Accuracy: 86.72999999999999
Correct Prediction: 8673
Wrong Prediction: 1327
Accuracy: 86.72999999999999


In [19]:
loaded_model = load_model('models/model.h5')

In [20]:
review = str(input('Movie Review: '))

In [22]:
import re

In [23]:
# Pre-process input
regex = re.compile(r'[^a-zA-Z\s]')
review = regex.sub('', review)
print('Cleaned: ', review)

words = review.split(' ')
filtered = [w for w in words if w not in english_stops]
filtered = ' '.join(filtered)
filtered = [filtered.lower()]

print('Filtered: ', filtered)

Cleaned:  This show was an amazing fresh  innovative idea in the s when it first aired The first  or  years were brilliant but things dropped off after that By  the show was not really funny anymore and its continued its decline further to the complete waste of time it is todaybr br Its truly disgraceful how far this show has fallen The writing is painfully bad the performances are almost as bad  if not for the mildly entertaining respite of the guesthosts this show probably wouldnt still be on the air I find it so hard to believe that the same creator that handselected the original cast also chose the band of hacks that followed How can one recognize such brilliance and then see fit to replace it with such mediocrity I felt I must give  stars out of respect for the original cast that made this show such a huge success As it is now the show is just awful I cant believe its still on the air
Filtered:  ['this show amazing fresh  innovative idea first aired the first   years brilliant thi

In [24]:
tokenize_words = token.texts_to_sequences(filtered)
tokenize_words = pad_sequences(tokenize_words, maxlen=max_length, padding='post', truncating='post')
print(tokenize_words)

[[    8    45   395  1359  4040   226    23  2909     2    23    68   421
     90  3364   771    45    14    72  1489  3508  5889   510   358    10
  34301   739   276 13038   131    45  2969     2   405  2222    19   272
    124    19  2550   344 15885    45   143 23830    57   801     1    80
    159   167  4434   120    87    22  2394  1016 10478  1532   407     5
   2341  3386    15  1044  4849  6630     1   348     1   110   107   310
   1056   120    87    24    45   545   921   105    45   277     1  2133
    167    57   801     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0     0     0
      0     0     0     0     0     0     0     0     0     0]]


In [25]:
result = loaded_model.predict(tokenize_words)
print(result)

1/1 [==============================] - 0s 323ms/step
[[0.00321356]]
1/1 [==============================] - 0s 323ms/step
[[0.00321356]]


In [26]:
if result >= 0.7:
    print('positive')
else:
    print('negative')

negative
